# Scratch Files to Jupyter Notebooks

This guide converts the Wi-Fi 7, UWB, and Bluetooth `# %%` scratch scripts into `.ipynb` notebooks. It preserves code-cell order, carries the introductory Markdown across, and adds Markdown headings from cell labels.

The original `.py` files remain unchanged.

## 1. Set Up the Development Environment

The project `.venv` already provides the Python kernel. Install `nbformat` for structured notebook creation; the Jupyter extension and `ipykernel` provide the Interactive Window and kernel.

In [ ]:
%pip install -q nbformat

from pathlib import Path
import ipykernel
import nbformat
import jupyter_core

print(f"ipykernel: {ipykernel.__version__}")
print(f"nbformat: {nbformat.__version__}")
print(f"Jupyter core: {jupyter_core.__version__}")

## 2. Create a Basic Jupyter Notebook

A notebook is structured data: metadata plus ordered Markdown and code cells. `nbformat` builds and validates that structure without hand-editing JSON.

In [ ]:
basic_notebook = nbformat.v4.new_notebook(metadata={
    "kernelspec": {
        "display_name": "Python 3.12.10 (.venv)",
        "language": "python",
        "name": "python3",
    },
    "language_info": {"name": "python"},
})
basic_notebook.cells = [
    nbformat.v4.new_markdown_cell("# Example notebook\nA Markdown introduction."),
    nbformat.v4.new_code_cell("value = 6 * 7\nprint(value)"),
]
nbformat.validate(basic_notebook)
print(f"Created an in-memory notebook with {len(basic_notebook.cells)} cells.")

## 3. Convert Scratch Code to Jupyter Cells

The converter recognizes VS Code `# %%` markers. A `# %% [markdown]` block becomes Markdown; labeled code-cell markers become a Markdown heading followed by the corresponding Python cell. Cell order and code text are preserved, and output notebooks are written beside the source files.

In [ ]:
import re

project_root = Path.cwd()
if not (project_root / "wifi7_scratch.py").is_file():
    project_root = project_root / "nr5g_demod"
if not (project_root / "wifi7_scratch.py").is_file():
    raise FileNotFoundError("Run this notebook from the workspace or nr5g_demod project folder")
project_root = project_root.resolve()
cell_marker = re.compile(r"^# %%\s*(.*)$")


def convert_python_scratch(source_path):
    if source_path.suffix != ".py":
        raise ValueError(f"Expected a Python scratch file: {source_path}")

    lines = source_path.read_text(encoding="utf-8").splitlines()
    markers = [(index, match.group(1).strip())
               for index, line in enumerate(lines)
               if (match := cell_marker.match(line))]
    if not markers:
        raise ValueError(f"No # %% cell markers found in {source_path}")

    notebook = nbformat.v4.new_notebook(metadata={
        "kernelspec": {
            "display_name": "Python 3.12.10 (.venv)",
            "language": "python",
            "name": "python3",
        },
        "language_info": {"name": "python"},
    })

    for marker_index, (start, label) in enumerate(markers):
        end = markers[marker_index + 1][0] if marker_index + 1 < len(markers) else len(lines)
        body = lines[start + 1:end]

        if label == "[markdown]":
            markdown_lines = [
                line[2:] if line.startswith("# ") else "" if line == "#" else line
                for line in body
            ]
            markdown = "\n".join(markdown_lines).strip()
            if markdown:
                notebook.cells.append(nbformat.v4.new_markdown_cell(markdown))
            continue

        if label:
            notebook.cells.append(nbformat.v4.new_markdown_cell(f"## {label}"))
        code = "\n".join(body).strip("\n")
        if code:
            notebook.cells.append(nbformat.v4.new_code_cell(code))

    nbformat.validate(notebook)
    output_path = source_path.with_suffix(".ipynb")
    nbformat.write(notebook, output_path)
    return output_path, len(notebook.cells)


scratch_files = [
    "wifi7_scratch.py",
    "uwb_scratch.py",
    "bluetooth_scratch.py",
]
converted = [convert_python_scratch(project_root / name) for name in scratch_files]
for notebook_path, cell_count in converted:
    print(f"{notebook_path.name}: {cell_count} cells")

## 4. Add Markdown Explanations to Notebook

The source overview block becomes a Markdown cell, and each labeled `# %%` section becomes a Markdown heading immediately before its code cell. The converter creates fresh code cells with no stale execution outputs.

In [ ]:
for notebook_path, _ in converted:
    notebook = nbformat.read(notebook_path, as_version=4)
    nbformat.validate(notebook)
    markdown_count = sum(cell.cell_type == "markdown" for cell in notebook.cells)
    code_count = sum(cell.cell_type == "code" for cell in notebook.cells)
    assert code_count > 0 and markdown_count > 0
    assert all(cell.execution_count is None and not cell.outputs
               for cell in notebook.cells if cell.cell_type == "code")
    print(f"Validated {notebook_path.name}: {markdown_count} Markdown, {code_count} code cells")